# Attention is All You Need 代码复现
复现 Transformer 原论文中提出的架构，运行在Google Colab提供的GPU上。

导入必须的包。

In [1]:
# Import some modules
# Data processing and visualization
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# Machine learning with PyTorch
import torch
from torch import nn
import torch.nn.functional as F
import torch.optim as optim

## Positional Encoding
先实现位置编码功能。

In [4]:
class PositionalEncoding(nn.Module): # input: [max_len, d_model]; output: [max_len, d_model]
    def __init__(self, max_len=1000, d_model=512, dropout=0.1):  # max_len表示位置编码的最大长度
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # 分母div_term
        div_term = torch.pow(10000, (torch.arange(0, d_model, 2, dtype=torch.float32) / d_model)) 
        # 分子position
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1) 

        # 计算PE
        self.PE = torch.zeros((max_len, d_model), dtype=torch.float32)
        self.PE[:, 0::2] = torch.sin(position / div_term)
        self.PE[:, 1::2] = torch.cos(position / div_term)
        # print(self.PE.shape)  # [max_len, d_model]

    def forward(self, X):  # X: [batch_size, seq_len, d_model]
        # [:X.size(1), :] 表示取前seq_len行，所有列 [seq_len, d_model]
        # 利用广播机制，将PE加到X上
        X = X + self.PE[:X.size(1), :].to(X.device)  
        return self.dropout(X)


## Scaled Dot-Product Attention
点积注意力的实现：$$ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$


In [5]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

    # Q: [batch_size, seq_len_q, d_k]; 
    # K: [batch_size, seq_len_k, d_k]; 
    # V: [batch_size, seq_len_v, d_v]; (seq_len_k == seq_len_v)
    # mask: [batch_size, seq_len_q, seq_len_k]
    def forward(self, Q, K, V, mask=None): 
        scores = torch.bmm(Q, torch.transpose(K, 1, 2)) # [batch_size, seq_len_q, seq_len_k]
        d_k = Q.size(-1)  # 最后一个维度的大小
        scores = scores / math.sqrt(d_k)  
        
        # 应用mask
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)  # 将mask中为0的位置对应的score设为一个很小的值
        # 计算注意力
        attn_w = F.softmax(scores, dim=-1)  # [batch_size, seq_len_q, seq_len_k]
        # 标准做法：在加权求和之前，对注意力权重进行 dropout，相当于随机“遮挡”掉一些注意力连接
        attn_w = self.dropout(attn_w)  
        # 加权求和 V
        attn = torch.bmm(attn_w, V) # [batch_size, seq_len_q, d_v]
        
        return attn, attn_w

## Multi-Head Attention
实现多头注意力机制。

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, num_heads=8, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"  # 确保可以均分
        self.dropout = nn.Dropout(p=dropout)
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 每个头的维度 512/8=64
        self.attention = ScaledDotProductAttention(dropout=dropout)
        # 准备线性层，输入X[batch_size, seq_len, d_model]
        # 这里输出维度也是d_model，是一个工程技巧
        # 是将8*(512->64)的8个线性变换合并成1*(512->512)的1个线性变换
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        
    # mask: [batch_size, seq_len_q, seq_len_k]
    def forward(self, Q, K, V, mask=None): 
        Q = self.W_Q(Q)  # [batch_size, seq_len_q, d_model]
        K = self.W_K(K)
        V = self.W_V(V)
        batch_size = Q.size(0)

        # 分割为多个头
        Q = Q.reshape(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2) # [batch_size, num_heads, seq_len_q, d_k]
        K = K.reshape(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = V.reshape(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # 合并batch_size和num_heads维度，方便计算注意力
        Q = Q.reshape(-1, Q.size(2), Q.size(3))  # [batch_size*num_heads, seq_len_q, d_k]
        K = K.reshape(-1, K.size(2), K.size(3))
        V = V.reshape(-1, V.size(2), V.size(3))

        # 处理mask
        if mask is not None:
            # 需要在维度0上扩展num_heads倍
            mask = mask.repeat_interleave(self.num_heads, dim=0)  # [batch_size*num_heads, seq_len_q, seq_len_k]
        # 计算注意力
        attn, attn_w = self.attention(Q, K, V, mask=mask)  # attn: [batch_size*num_heads, seq_len_q, d_k]

        # 恢复形状
        attn = attn.reshape(batch_size, self.num_heads, -1, self.d_k).transpose(1, 2)  # [batch_size, seq_len_q, num_heads, d_k]
        attn = attn.reshape(batch_size, -1, self.d_model)  # [batch_size, seq_len_q, d_model]
        output = self.W_O(attn)  # 线性变换输出

        attn_w = attn_w.reshape(batch_size, self.num_heads, attn_w.size(1), -1)  # [batch_size, num_heads, seq_len_q, seq_len_k]

        return output, attn_w

## Position-wise Feed-Forward Networks
基于位置的前馈网络(FFN)。

In [ ]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model=512, d_ff=2048, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, X):  # X: [batch_size, seq_len, d_model]
        return self.fc2(self.dropout(F.relu(self.fc1(X))))
        

## Encoder
### Encoder Layer
有了多头注意力块和前馈神经网络，就可以组装Encoder块了。

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model=512, d_ff=2048, num_heads=8, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads, dropout=dropout)
        self.ffn = PositionWiseFeedForward(d_model=d_model, d_ff=d_ff, dropout=dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, X, mask=None):  # X: [batch_size, seq_len, d_model]
        # 此处采用的是Post-LN (后归一化)结构，近年来有研究表明Pre-LN (前归一化)结构在深层Transformer中更稳定
        # 多头注意力子层
        attn, attn_w = self.mha(X, X, X, mask=mask)
        X = self.norm1(X + self.dropout(attn))  # 残差连接 + 层归一化
        # 前馈神经网络子层
        ffn_output = self.ffn(X)
        X = self.norm2(X + self.dropout(ffn_output))  # 残差连接

        return X, attn_w

### Encoder Class
已经实现了Encoder Layer，现在只需要堆叠多个Encoders，并且添加 Embedding 和 Positional Encoding，就可以构筑完整的Transformer编码器。

> The encoder is composed of a stack of N = 6 identical layers.

In [5]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, stack_layers = 6, d_model=512, d_ff=2048, num_heads=8, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(max_len=1000, d_model=d_model, dropout=dropout)
        self.encoderLayers = nn.ModuleList([EncoderLayer(d_model=d_model, d_ff=d_ff, num_heads=num_heads, dropout=dropout) for _ in range(stack_layers)])
    
    def forward(self, X, mask=None):  # X: [batch_size, seq_len]
        # Embedding and Positional Encoding
        # "In the embedding layers, we multiply those weights by sqrt(dmodel)."
        X = self.embedding(X) * math.sqrt(self.embedding.embedding_dim)  # [batch_size, seq_len, d_model]
        X = self.pos_encoding(X)  # Add positional encoding

        # Pass through each Encoder layer
        attn_weights = []  # 存储每层的注意力权重
        for layer in self.encoderLayers:  # 通过每一层
            X, attn_w = layer(X, mask=mask)
            attn_weights.append(attn_w)

        return X, attn_weights  # X: [batch_size, seq_len, d_model]，输出的上下文向量

## Decoder
### Decoder Layer

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model=512, d_ff=2048, num_heads=8, dropout=0.1):
        super().__init__()
        self.masked_mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads, dropout=dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads, dropout=dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = PositionWiseFeedForward(d_model=d_model, d_ff=d_ff, dropout=dropout)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(p=dropout)